# USPTO 短语匹配 — 微调训练（Kaggle kernel 入口，单文件）

本 notebook 即推送到 Kaggle 的运行产物（Kaggle 只执行一个 notebook）。
内部按「模块」分区，自上而下顺序执行即可：
1. **模块 1 环境配置** — 数据/模型开关、路径、设备探测
2. **模块 2 文件服务** — 依据配置实现数据读取 / 模型存取
3. **模块 3 训练入口** — 确认配置并载入数据（在此之上接微调骨架）

**开关 `USE_KAGGLE`**（见模块 1）：`True`=Kaggle 云端读写；`False`=本地读写；
`None`=自动探测 `/kaggle/input`。设备 `DEVICE` 自动检测 cuda/mps/cpu。

## 模块 1：环境配置（数据/模型开关 + 路径 + 设备探测）

In [ ]:
# ============================================================
# 模块 1：环境配置（数据/模型开关 + 路径 + 设备探测）
# 翻开关：把下面 USE_KAGGLE 改成 True / False 即可强制切换。
# ============================================================
import os
from pathlib import Path

# 手动开关：None=自动探测；True=Kaggle 云端；False=本地
USE_KAGGLE = None
COMPETITION_INPUT_NAME = 'competitions/us-patent-phrase-to-phrase-matching'


def _resolve_use_kaggle() -> bool:
    env = os.environ.get('USE_KAGGLE', '').strip().lower()
    if env in ('1', 'true', 'yes', 'on'):
        return True
    if env in ('0', 'false', 'no', 'off'):
        return False
    if USE_KAGGLE is not None:
        return bool(USE_KAGGLE)
    return os.path.isdir('/kaggle/input')


ON_KAGGLE = _resolve_use_kaggle()


def _detect_device() -> str:
    '''实时探测可用计算设备：cuda > mps > cpu。
    torch 未安装时安全回退为 cpu（仅返回字符串，不依赖 torch）。'''
    try:
        import torch
        if torch.cuda.is_available():
            return 'cuda'
        mps = getattr(torch.backends, 'mps', None)
        if mps is not None and mps.is_available():
            return 'mps'
    except ImportError:
        pass
    return 'cpu'


# 当前计算设备：'cuda' / 'mps' / 'cpu'
DEVICE = _detect_device()


def get_device():
    '''返回 torch.device（需已安装 torch）；否则回退为字符串 DEVICE。
    训练代码可直接：model.to(get_device())。'''
    try:
        import torch
        return torch.device(DEVICE)
    except ImportError:
        return DEVICE


def _find_local_data() -> Path:
    '''本地模式下，向上查找含 train.csv 的 data 目录。'''
    base = Path(os.getcwd())
    for cand in [base, *base.parents]:
        if (cand / 'data' / 'train.csv').exists():
            return cand / 'data'
    return base / 'data'


LOCAL_DATA_DIR = _find_local_data()
LOCAL_MODEL_DIR = LOCAL_DATA_DIR.parent / 'train' / 'outputs'

KAGGLE_DATA_DIR = f'/kaggle/input/{COMPETITION_INPUT_NAME}'
KAGGLE_MODEL_DIR = '/kaggle/working'

DATA_DIR = KAGGLE_DATA_DIR if ON_KAGGLE else str(LOCAL_DATA_DIR)
MODEL_DIR = KAGGLE_MODEL_DIR if ON_KAGGLE else str(LOCAL_MODEL_DIR)


def summary() -> dict:
    '''返回当前开关、设备与路径概览。'''
    train_csv = (
        os.path.join(DATA_DIR, 'train.csv')
        if ON_KAGGLE
        else str(LOCAL_DATA_DIR / 'train.csv')
    )
    return {
        'ON_KAGGLE': ON_KAGGLE,
        'DEVICE': DEVICE,
        'DATA_DIR': DATA_DIR,
        'MODEL_DIR': MODEL_DIR,
        'train_csv': train_csv,
    }

## 模块 2：文件服务（数据读取 + 模型存取）

依赖模块 1 已定义的 `ON_KAGGLE` / `DATA_DIR` / `MODEL_DIR`。

In [34]:
# ============================================================
# 模块 2：文件服务（数据读取 + 模型存取）
# 依赖上方模块 1 已定义的 ON_KAGGLE / DATA_DIR / MODEL_DIR。
# ============================================================
import os


def data_path(filename: str) -> str:
    '''解析数据文件绝对路径。
    Kaggle 模式下：依次尝试 DATA_DIR、/kaggle/input 根目录，并递归搜索
    /kaggle/input 下任意子目录，找到即返回（不依赖固定目录名）。
    本地模式下：仅在 DATA_DIR 中查找。
    '''
    if ON_KAGGLE:
        candidates = [
            os.path.join(DATA_DIR, filename),
            os.path.join('/kaggle/input', filename),
        ]
        if os.path.isdir('/kaggle/input'):
            for root, _dirs, files in os.walk('/kaggle/input'):
                if filename in files:
                    candidates.append(os.path.join(root, filename))
        for c in candidates:
            if os.path.exists(c):
                return c
        top = os.listdir('/kaggle/input') if os.path.isdir('/kaggle/input') else '(目录不存在)'
        raise FileNotFoundError(
            f'Kaggle 上未找到 {filename}。\n'
            f'/kaggle/input 顶层内容: {top}\n'
            '请确认已通过「+ Add data」把比赛数据集附加到 kernel。'
        )
    cand = os.path.join(DATA_DIR, filename)
    if not os.path.exists(cand):
        raise FileNotFoundError(f'本地数据缺失: {cand}')
    return cand


def model_path(filename: str) -> str:
    '''解析模型保存/加载路径，并确保目录已创建。'''
    os.makedirs(MODEL_DIR, exist_ok=True)
    return os.path.join(MODEL_DIR, filename)


def load_train():
    '''读取训练集。'''
    import pandas as pd
    return pd.read_csv(data_path('train.csv'))


def load_test():
    '''读取测试集。'''
    import pandas as pd
    return pd.read_csv(data_path('test.csv'))


def save_model(model, filename: str) -> str:
    '''保存模型到当前 MODEL_DIR（本地或 Kaggle 云端）。'''
    path = model_path(filename)
    try:
        import torch
        torch.save(model, path)
    except ImportError:
        import pickle
        with open(path, 'wb') as f:
            pickle.dump(model, f)
    return path


def load_model(filename: str):
    '''从当前 MODEL_DIR 加载模型，与 save_model 对称。'''
    path = model_path(filename)
    try:
        import torch
        return torch.load(path, map_location='cpu')
    except ImportError:
        import pickle
        with open(path, 'rb') as f:
            return pickle.load(f)

## 模块 3：训练入口（示例 / 微调骨架入口）

In [35]:
# ============================================================
# 模块 3：训练入口
# 确认当前开关 / 设备 / 路径，并载入训练集（在此之上接微调骨架）。
# ============================================================
print(summary())
print('get_device() ->', get_device())

df = load_train()
print('train shape:', df.shape)
df.head()

{'ON_KAGGLE': False, 'DEVICE': 'mps', 'DATA_DIR': '/Volumes/zhitai2/git/python/USPTO/data', 'MODEL_DIR': '/Volumes/zhitai2/git/python/USPTO/train/outputs', 'train_csv': '/Volumes/zhitai2/git/python/USPTO/data/train.csv'}
get_device() -> mps
train shape: (36473, 5)


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


## 模块 4：数据可视化（EDA，基于 seaborn）

把 `explore.py` 的可视化能力并入单文件 notebook（本单元格定义绘图函数）。
依赖模块 1/2 的 `load_train()` 与模块 3 已载入的 `df`。
图内标签使用英文，避免 matplotlib 默认字体缺中文导致乱码；backend 设为 Agg，
无图形界面也能把图存为文件（Kaggle 上落在 /kaggle/working，可随 kernel 产出下载）。

In [36]:
# ============================================================
# 模块 4：数据可视化（基于 seaborn）
# 把 explore.py 的绘图与分析辅助函数并入单文件 notebook。
# ============================================================
import os
import pandas as pd
import matplotlib

if os.environ.get('MPLBACKEND') is None:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')


def _non_empty(d):
    if d is None or len(d) == 0:
        raise ValueError('数据为空，无法分析/绘图。')


def score_distribution(d):
    vc = d['score'].value_counts(dropna=False).sort_index()
    out = pd.DataFrame({'score': vc.index, 'count': vc.values})
    out['proportion'] = (out['count'] / len(d)).round(4)
    return out


def context_distribution(d, top_n=20):
    vc = d['context'].value_counts(dropna=False)
    out = pd.DataFrame({'context': vc.index, 'count': vc.values})
    out['proportion'] = (out['count'] / len(d)).round(4)
    return out.head(top_n) if top_n else out


def score_by_context(d):
    g = d.groupby('context')['score']
    out = pd.DataFrame({
        'n': g.size(),
        'mean_score': g.mean().round(4),
        'median_score': g.median(),
        'min_score': g.min(),
        'max_score': g.max(),
    }).sort_values('n', ascending=False)
    return out


def _tokenize(text):
    return set(str(text).lower().split())


def _jaccard(a, t):
    a, t = _tokenize(a), _tokenize(t)
    union = len(a | t)
    if union == 0:
        return 0.0
    return len(a & t) / union


def token_overlap(d):
    jac = d.apply(lambda r: _jaccard(r['anchor'], r['target']), axis=1)
    tmp = pd.DataFrame({'score': d['score'].to_numpy(), 'jaccard': jac.to_numpy()})
    g = tmp.groupby('score')['jaccard']
    return pd.DataFrame({'mean': g.mean().round(4), 'median': g.median().round(4), 'count': g.size()})


def _finalize(fig, save_path, show):
    if save_path:
        os.makedirs(os.path.dirname(os.path.abspath(save_path)), exist_ok=True)
        fig.savefig(save_path, dpi=120, bbox_inches='tight')
    if show:
        try:
            plt.show()
        except Exception:
            pass
    plt.close(fig)
    return fig


def plot_score_distribution(d, save_path=None, show=False):
    _non_empty(d)
    dist = score_distribution(d)
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=dist, x='score', y='count', ax=ax, color='#4C72B0')
    for _, row in dist.iterrows():
        ax.text(float(row['score']), float(row['count']), f"{float(row['proportion']):.1%}", ha='center', va='bottom', fontsize=9)
    ax.set_title('Score Distribution (label imbalance)')
    ax.set_xlabel('score')
    ax.set_ylabel('count')
    return _finalize(fig, save_path, show)


def plot_context_distribution(d, top_n=20, save_path=None, show=False):
    _non_empty(d)
    dist = context_distribution(d, top_n)
    fig, ax = plt.subplots(figsize=(9, max(4, top_n * 0.32)))
    sns.barplot(data=dist, x='count', y='context', ax=ax, color='#55A868')
    ax.set_title(f'Top-{top_n} Context (CPC class) by sample count')
    ax.set_xlabel('count')
    ax.set_ylabel('context')
    return _finalize(fig, save_path, show)


def plot_score_by_context(d, top_n=20, save_path=None, show=False):
    _non_empty(d)
    sbc = score_by_context(d).head(top_n).sort_values('mean_score')
    fig, ax = plt.subplots(figsize=(9, max(4, top_n * 0.32)))
    sns.barplot(data=sbc, x='mean_score', y=sbc.index, ax=ax, color='#C44E52')
    ax.set_title(f'Mean score by Top-{top_n} context')
    ax.set_xlabel('mean score')
    ax.set_ylabel('context')
    return _finalize(fig, save_path, show)


def plot_length_distributions(d, save_path=None, show=False):
    _non_empty(d)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, col, color in zip(axes, ('anchor', 'target'), ('#4C72B0', '#DD8452')):
        lens = d[col].astype(str).str.len()
        sns.histplot(lens, ax=ax, bins=40, color=color, kde=False)
        ax.axvline(lens.quantile(0.95), color='red', linestyle='--', label='p95')
        ax.set_title(f'{col} length (chars)')
        ax.set_xlabel('char length')
        ax.set_ylabel('count')
        ax.legend()
    fig.tight_layout()
    return _finalize(fig, save_path, show)


def plot_token_overlap(d, save_path=None, show=False):
    _non_empty(d)
    to = token_overlap(d).reset_index()
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.barplot(data=to, x='score', y='mean', ax=ax, color='#8172B3')
    for _, row in to.iterrows():
        ax.text(float(row['score']), float(row['mean']), f"{float(row['mean']):.2f}", ha='center', va='bottom', fontsize=9)
    ax.set_title('Mean token-overlap (Jaccard) by score')
    ax.set_xlabel('score')
    ax.set_ylabel('mean Jaccard')
    return _finalize(fig, save_path, show)


def plot_all(d, out_dir='figures'):
    _non_empty(d)
    names = ['score_distribution', 'context_distribution', 'score_by_context', 'length_distributions', 'token_overlap']
    paths = []
    for name in names:
        save_path = os.path.join(out_dir, f'{name}.png')
        globals()[f'plot_{name}'](d, save_path=save_path, show=False)
        paths.append(save_path)
    return paths

## 模块 5：生成可视化（保存为图片）

复用模块 3 已载入的 `df`，把 5 张图保存到 `MODEL_DIR/figures`（Kaggle 上即 `/kaggle/working/figures`，
可随 kernel 产出下载；本地即 `train/outputs/figures`）。

In [37]:
# ============================================================
# 模块 5：生成全部可视化并保存
# ============================================================
fig_dir = os.path.join(MODEL_DIR, 'figures')
paths = plot_all(df, out_dir=fig_dir)
print('可视化已保存到:', fig_dir)
for p in paths:
    size = os.path.getsize(p) if os.path.exists(p) else 'MISSING'
    print(' -', p, size)

可视化已保存到: /Volumes/zhitai2/git/python/USPTO/train/outputs/figures
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/score_distribution.png 28927
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/context_distribution.png 40246
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/score_by_context.png 41372
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/length_distributions.png 41079
 - /Volumes/zhitai2/git/python/USPTO/train/outputs/figures/token_overlap.png 26353


## 模块 6：模型选择配置

通过 `MODEL_KEY` 在 `deberta-v3-small` / `deberta-v3-base` 之间切换，后期可做精度/速度对比。
`MODEL_HF_ID` 默认是 HuggingFace 仓库名；若本地 `MODEL_DIR/pretrained/<key>` 已存在权重
（用 `modelscope` 或 `hf-mirror` 下载），则自动改用本地目录**离线加载**，避免联网拉取。

In [38]:
# ============================================================
# 模块 6：模型选择配置
# 切换 MODEL_KEY 即可选用不同尺寸的 DeBERTa-v3（用于对比）。
# ============================================================
import os

# 本地访问 HuggingFace 慢/不通时走国内镜像；Kaggle 上直连官方
if not ON_KAGGLE:
    os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

MODEL_KEY = 'deberta-v3-small'   # 亦可改 'deberta-v3-base' 做对比
MODEL_ID_MAP = {
    'deberta-v3-small': 'microsoft/deberta-v3-small',
    'deberta-v3-base':  'microsoft/deberta-v3-base',
}

# 本地预训练权重缓存目录（用 modelscope / hf-mirror 下载到此，离线加载）
PRETRAINED_DIR = os.path.join(MODEL_DIR, 'pretrained')


def resolve_model_id(key):
    '''本地 ./pretrained/<key> 存在则优先用本地目录，否则回退 HuggingFace 仓库名。'''
    local = os.path.join(PRETRAINED_DIR, key)
    if os.path.isfile(os.path.join(local, 'config.json')):
        return local
    return MODEL_ID_MAP[key]


MODEL_HF_ID = resolve_model_id(MODEL_KEY)
FINETUNED_FILENAME = f'{MODEL_KEY}_finetuned.pt'


def finetuned_model_exists() -> bool:
    '''检测当前模型是否已有微调产物（落在本地的 MODEL_DIR）。'''
    return os.path.exists(model_path(FINETUNED_FILENAME))


print('选用模型:', MODEL_KEY, '->', MODEL_HF_ID)
print('本地已有微调模型:', finetuned_model_exists())

选用模型: deberta-v3-small -> microsoft/deberta-v3-small
本地已有微调模型: False


## 模块 7：数据分割（训练 / 验证 / 测试 = 8:1:1）

固定随机种子以保证每次分割一致，方便 small / base、微调前/后的公平对比。

In [39]:
# ============================================================
# 模块 7：数据分割（8:1:1，固定种子保证可复现）
# ============================================================
import numpy as np


def split_train_val_test(df, val_ratio=0.1, test_ratio=0.1, seed=42):
    '''按 8:1:1 随机分割，返回 (train_df, val_df, test_df)。固定种子可复现。'''
    rng = np.random.default_rng(seed)
    n = len(df)
    idx = rng.permutation(n)
    n_val = int(round(n * val_ratio))
    n_test = int(round(n * test_ratio))
    test_idx = idx[:n_test]
    val_idx = idx[n_test:n_test + n_val]
    train_idx = idx[n_test + n_val:]
    return (df.iloc[train_idx].reset_index(drop=True),
            df.iloc[val_idx].reset_index(drop=True),
            df.iloc[test_idx].reset_index(drop=True))


train_df, val_df, test_df = split_train_val_test(df)
print('分割完成 ->',
      'train:', train_df.shape[0],
      '| val:', val_df.shape[0],
      '| test:', test_df.shape[0])

分割完成 -> train: 29179 | val: 3647 | test: 3647


## 模块 8：模型与分词器构建（DeBERTa 编码器 + 回归头）

定义 `PhraseMatcher`：DeBERTa 编码器输出 pooled 表示，接 Dropout + Linear(1) + sigmoid，
把分数约束到 [0,1]。同时提供 `predict_scores`（批量推理）与 `pearson`（评估）工具函数。
本模块只定义与构造，**不含训练循环**（训练在后续模块实现）。

In [40]:
# ============================================================
# 模块 8：模型与分词器构建（仅定义/构造，不做训练）
# ============================================================
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModel


class PhraseMatcher(nn.Module):
    '''DeBERTa 编码器 + 回归头：输入拼接文本，输出 [0,1] 的相似度得分。'''
    def __init__(self, model_id, hidden_dropout=0.1):
        super().__init__()
        # torch_dtype=float32：强制 fp32，避免与 fp32 回归头出现 Half/Float 不匹配
        self.encoder = AutoModel.from_pretrained(model_id, torch_dtype=torch.float32)
        hidden = self.encoder.config.hidden_size
        self.head = nn.Sequential(nn.Dropout(hidden_dropout), nn.Linear(hidden, 1))

    def forward(self, **inputs):
        # DeBERTa 不使用 token_type_ids，只传 input_ids / attention_mask
        out = self.encoder(**inputs).last_hidden_state
        cls = out[:, 0]                 # 取首 token 作为句表示
        return torch.sigmoid(self.head(cls).squeeze(-1))


def build_texts(df, tokenizer):
    '''拼接 anchor [SEP] target [SEP] context 为模型输入文本。'''
    sep = tokenizer.sep_token or ' '
    return (df['anchor'].astype(str) + sep + df['target'].astype(str)
            + sep + df['context'].astype(str)).tolist()


def predict_scores(model, tokenizer, df, device, max_len=128, batch_size=64):
    '''对 df 批量推理，返回 numpy 数组的预测分数（[0,1]）。'''
    model.eval()
    texts = build_texts(df, tokenizer)
    preds = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_len, return_tensors='pt')
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            s = model(**enc)
        preds.append(s.cpu().numpy())
    return np.concatenate(preds)


def pearson(y_true, y_pred):
    '''Pearson 相关系数（官方评测指标）。'''
    yt = np.asarray(y_true, float)
    yp = np.asarray(y_pred, float)
    if yt.std() == 0 or yp.std() == 0:
        return float('nan')
    return float(np.corrcoef(yt, yp)[0, 1])

## 模块 9：原生模型基线预测（未微调，供后续对比）

用**未微调**的 DeBERTa（编码器预训练权重 + 随机初始化回归头）在验证/测试集上推理，
作为微调前的"地板"参考：其 Pearson 应接近 0。把预测结果存到 `MODEL_DIR/baselines`，
微调完成后用同样的 `predict_scores` 跑一遍，即可量化微调带来的提升，并公平对比 small/base。

In [41]:
# ============================================================
# 模块 9：原生模型基线预测（未微调，仅前向推理，不训练）
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_ID)
model = PhraseMatcher(MODEL_HF_ID).float().to(get_device())

val_pred = predict_scores(model, tokenizer, val_df, get_device())
test_pred = predict_scores(model, tokenizer, test_df, get_device())

val_pe = pearson(val_df['score'], val_pred)
test_pe = pearson(test_df['score'], test_pred)
print(f'基线(未微调) Pearson -> val: {val_pe:.4f} | test: {test_pe:.4f}')

# 保存基线预测，供后续微调后对比
base_dir = os.path.join(MODEL_DIR, 'baselines')
os.makedirs(base_dir, exist_ok=True)
np.save(os.path.join(base_dir, f'{MODEL_KEY}_val_pred.npy'), val_pred)
np.save(os.path.join(base_dir, f'{MODEL_KEY}_test_pred.npy'), test_pred)
print('基线预测已保存至:', base_dir)

'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
Retrying in 8s [Retry 4/5].
'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
Retrying in 8s [Retry 5/5].
'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
'timed out' thrown while requesting HEAD https://huggingface.co/microsoft/deberta-v3-small/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'timed out' thrown w

OSError: We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.

## 模块 9（补）：原生基线预测对比（CSV + 可视化）

用 8:1:1 分割出的**测试集**（`test_df`）做原生模型预测，把「真实 label vs 预测值」存成 CSV 便于观测，并画对比图。

In [ ]:
# ============================================================
# 模块 9（补）：测试集 真实值 vs 基线预测 —— CSV + 可视化
# （接在模块 9 之后；test_df / test_pred / base_dir 已在上方计算）
# ============================================================
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# 1) 保存测试集 真实值 vs 预测值 到 CSV
compare = pd.DataFrame({
    'id': test_df['id'].values,
    'anchor': test_df['anchor'].values,
    'target': test_df['target'].values,
    'context': test_df['context'].values,
    'true_score': test_df['score'].values,
    'pred_score': test_pred,
})
csv_path = os.path.join(base_dir, f'{MODEL_KEY}_test_compare.csv')
compare.to_csv(csv_path, index=False)
print('测试集对比 CSV 已保存:', csv_path)
print(compare.head())

# 2) 可视化：真实 vs 预测（散点 + 分布对比）
pe = pearson(compare['true_score'], compare['pred_score'])
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(compare['true_score'], compare['pred_score'], alpha=0.25, s=12)
axes[0].set_xlabel('True score')
axes[0].set_ylabel('Predicted score (native baseline)')
axes[0].set_title(f'True vs Pred (test)\nPearson={pe:.4f}')

sns.histplot(compare['true_score'], color='C0', label='true', bins=20,
             ax=axes[1], stat='density', alpha=0.5)
sns.histplot(compare['pred_score'], color='C1', label='pred', bins=20,
             ax=axes[1], stat='density', alpha=0.5)
axes[1].set_xlabel('score')
axes[1].set_ylabel('density')
axes[1].set_title('Distribution: true vs pred (test)')
axes[1].legend()

plt.tight_layout()
fig_dir = os.path.join(MODEL_DIR, 'figures')
os.makedirs(fig_dir, exist_ok=True)
fig_path = os.path.join(fig_dir, f'{MODEL_KEY}_baseline_compare.png')
fig.savefig(fig_path, dpi=120)
plt.close(fig)
print('对比图已保存:', fig_path)